### dbt (data build tool)

- is a transformation (T in ELT) that applies software engineering concepts (git, testing) to modular SQL.

**Use Cases**: Cleaning raw data, building analytics-ready tables, and managing data lineage.

Traditionally, the data transformation process was handled by long, monolithic SQL script. Long SQL scripts are hard to debug with no lineage tracking and testing. 

dbt provides:
- Modularity : instead of gient blocks, create small, reusable models.
- Automated documentation & lineage: dbt docs generate
- Built-in testing: yml configs to ensure data is healthy

#### dbt Project Components
- project config
- data sources and destinations
- SQL queries
- templates
- documentation

`dbt init` 
- params: project name & db/warehouse type
- create top lvl project folder and all structure required
> - dbt init test_project <br>
> - choose database > [1] duckdb 

**YML config files**

`dbt_project.yml`
- project global config
- contains contents related to full project such as 
    - project name/version
    - directory locations
***
`profiles.yml` : 
- credentials such as database host, port, user, password (Usually kept outside the git repo for security)
- defines dbt profiles (environments) such as dev, staging/testing, prod

```yaml
    marketing_campaign_results:
        outputs:
            dev:
                type: duckdb
                path: dbt.duckdb
            prod:
                type: snowflake
                ...
        target: dev
```
***
`model_properties.yml`
- should reside in models/subdirectory but filename can be anything 
- contains settings that reference models
- description, doc details, testing etc.,

`dbt debug`: test if any error in configuration
<br>*note*: runtime environment needs to be in the dbt project folder

`dbt run`
- run whenever data model changes or data process needs to be materalized
- output provides details on success/failure of steps involved
- `./datacheck` to check sample content and len from data warehouse
- `dbt run --select filename` to run only selected model file
- `dbt run --select +agg_daily_cc_riders` Run this model AND everything upstream that it depends on

`dbt test`
- checks table for errors (like `not_null`, `unique`, etc)

#### Workflow for dbt
1. create project with `db init`
2. define configuration `profiles.yml`
3. Create user models / templates
4. instantiate models with `dbt run`

**The Workflow: Source to Warehouse**
The industry standard is the Medallion Architecture. Here is how the data travels:

| Stage  | dbt Layer     | What happens? |
|--------|---------------|---------------|
| Source | Raw Data      | Your .parquet file or an external API. |
| Bronze | staging       | dbt run --s stg_model. Light cleaning (renaming columns, fixing data types). |
| Silver | intermediate  | We join tables together (e.g., joining Taxi Rides with Weather data). |
| Gold   | marts         | dbt run --s fct_model. Final business logic (e.g., "Total Revenue per Day"). |

The "Standard" Workflow loop:
1. Write SQL in a new model.
2. Run dbt run --select my_new_model to build it.
3. Preview the data in your warehouse.
4. Write tests in the .yml file.
5. Run dbt test --select my_new_model to make sure it's clean.

#### Updating dbt models
- easy to make changes without needing to write new queries/models from scratch each time

**Update Workflow**
- checkout from source control: `git clone dbt_project`
- update query from selected model
- regenerate with `dbt run` or `dbt run -f` 

- some updates may require changes in `dbt_project.yml` or `model_properties.yml`

#### Creating and Generating dbt documentation `dbt docs` 

- can provide documentation with model definitions
- can add documentation about columns within models
- automatically show data lineage/DAG
- view generated warehouse info (col data types, data sizes)
- run after `dbt run` to create a doc website based on the project
- run `dbt docs generate` to generate documentation 
- `dbt docs serve` to start a local webserver for doc website
- can be hosted on other services (dbt cloud, Amazon S3, Nginx, apache etc)

```yaml
version: 2
models:
- name: taxi_rides_raw
  description: Yellow Taxi raw data
  access: public
- name: avg_fare_per_day
  description: Average ride per day
  access: public 
```


### Jinja Templates
- to simplify creation of dbt model and other objects

**Jinja Functions**
- ref: access models by name
- config: access config settings
- docs: access documentation info

**Why Jinja**
> without jinja
```sql
SELECT 
    COALESCE(start_date, '2025-01-01') AS start_date, 
    COALESCE(update_date, '2025-01-01') AS update_date,
    COALESCE(end_date, '2025-01-01') AS end_date
FROM Events
```
> with jinja
```sql
SELECT
{% for column in ['start_date', 'update_date', 'end_date'] %}
COALESCE({{column}}, '2025-01-01') AS {{column}}
{% endfor %}
FROM Events
```

#### Hierarchical models

- hierarchy: represents dependency within a dbt project (a DAG or lineage graph)
- defined using jinja template in model files

for example: <br>
if : `tableA` >> `tableB` and `tableC`
then: dbt builds `tableA` first

without this lineage graph, dbt creates entries in alphabetical order and tasks with dependency will fail.

**Example**
- `ref` function
    - `{{ ref('model_name') }}` in SQL
    - when we do dbt run, it gets replaced with actual table

#### What happens when we run `dbt run`

1. Compilation
- dbt looks at all .sql file to build DAG 
- resolves Jinja template code (if any)
- wraps entire SQL inside `CREATE TABLE` statement and use filename of our .sql file as table name
- inside `./target/compiled` folder it generates temp file 

2. Connection & Execution
- checks `profiles.yml` to find keys to data warehouse, opens a connection and send generated SQL code to database
- database engine does the reading of .parquet file from disk and loading it to structured table

3. Materialization Strategy
- by default, dbt will create a `View` only if not specified
- if we want actual data to be stored on disk, add `{{ config(materialized='table') }}` on top of .sql file.

4. Manifest Update
- dbt then updates the file `target/manifest.json` recording build timestamp.